# Week 2 · Regional Corridors & Distance Bands
**E-Commerce Delivery & Logistics Analysis** · Owner: Akhila Nair · Branch: `feature/akhila-regional-analysis` · Issue #4

**Mission:** quantify geographic disparities in delivery performance: intrastate vs. interstate fulfilment, destination-state delay rates, the most problematic shipping corridors, and the effect of distance.

**Analytical standards applied**
- Raw data in `data/raw/` is never modified; analysis uses `data/interim/logistics_analytical_dataset.csv`.
- Population = *eligible* delivered orders (`is_eligible_otd == 1`).
- Every rate is reported together with its order count; groups with fewer than **20 orders** are excluded from rankings.
- Findings describe associations, not causes.

## 1. Setup & data loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.options.display.float_format = "{:,.2f}".format

# --- Project paths (works whether the notebook is run from /notebooks or the repo root) ---
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"          # read-only: never modified
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Analysis parameters ---
MIN_ORDERS = 20                                   # minimum sample size for any group comparison
PRIMARY, ACCENT, NEUTRAL = "#1F4E79", "#C0392B", "#9AA5B1"

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two sets of coordinates."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def add_distance_km(frame):
    """Add seller->customer distance (km) using the mean coordinates of each ZIP prefix.

    The analytical dataset has no coordinates, so they are derived from the raw geolocation and
    sellers tables (read-only). Distance is therefore an approximation at ZIP-prefix level.
    """
    geo = pd.read_csv(DATA_RAW / "olist_geolocation_dataset.csv")
    centroid = geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean()
    sellers = pd.read_csv(DATA_RAW / "olist_sellers_dataset.csv")[["seller_id", "seller_zip_code_prefix"]]

    out = frame.merge(sellers, on="seller_id", how="left")
    out = out.merge(centroid.add_prefix("cust_"), left_on="customer_zip_code_prefix", right_index=True, how="left")
    out = out.merge(centroid.add_prefix("sell_"), left_on="seller_zip_code_prefix", right_index=True, how="left")
    out["distance_km"] = haversine_km(out["sell_geolocation_lat"], out["sell_geolocation_lng"],
                                      out["cust_geolocation_lat"], out["cust_geolocation_lng"])
    return out.drop(columns=[c for c in out.columns if c.startswith(("cust_geo", "sell_geo"))])


BAND_EDGES = [0, 100, 300, 600, 1000, 1500, 2000, np.inf]
BAND_LABELS = ["<100 km", "100-300 km", "300-600 km", "600-1,000 km", "1,000-1,500 km", "1,500-2,000 km", ">2,000 km"]


def delay_rate(x):
    """Share of orders delivered late, in %."""
    return (x == "delayed").mean() * 100

In [ ]:
df = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "logistics_analytical_dataset.csv")
valid = df[df["is_eligible_otd"] == 1].copy()

national_delay = delay_rate(valid["delivery_outcome"])
print(f"All orders in dataset           : {len(df):>8,}")
print(f"Eligible delivered orders       : {len(valid):>8,}")
print(f"Destination states (customers)  : {valid['customer_state'].nunique():>8}")
print(f"Origin states (sellers)         : {valid['seller_state'].nunique():>8}")
print(f"National delay rate (baseline)  : {national_delay:>8.2f}%")

## 2. Intrastate vs. interstate fulfilment
*Question: how much slower, later and more expensive is an order when the seller and customer are in different states?*

In [ ]:
intra_vs_inter = valid.groupby("is_interstate").agg(
    order_count=("order_id", "count"),
    median_duration_days=("total_delivery_duration_days", "median"),
    delay_rate_pct=("delivery_outcome", delay_rate),
    avg_freight=("total_freight_value", "mean"),
)
intra_vs_inter.index = intra_vs_inter.index.map({False: "Intrastate", True: "Interstate"})
intra_vs_inter["share_of_orders_pct"] = intra_vs_inter["order_count"] / intra_vs_inter["order_count"].sum() * 100
display(intra_vs_inter)

gap = intra_vs_inter.loc["Interstate"] - intra_vs_inter.loc["Intrastate"]
print(f"Interstate vs intrastate: +{gap['median_duration_days']:.1f} median days, "
      f"+{gap['delay_rate_pct']:.1f} pp delay rate, +{gap['avg_freight']:.2f} average freight")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
metrics = [("median_duration_days", "Median delivery duration (days)"),
           ("delay_rate_pct", "Delay rate (%)"),
           ("avg_freight", "Average freight value (BRL)")]
for ax, (col, title) in zip(axes, metrics):
    bars = ax.bar(intra_vs_inter.index, intra_vs_inter[col], color=[NEUTRAL, PRIMARY], width=0.55)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.bar_label(bars, fmt="%.1f", padding=3)
    ax.set_ylim(0, intra_vs_inter[col].max() * 1.18)
for ax, n in zip(axes, [None] * 3):
    ax.set_xlabel("")
fig.suptitle(f"Intrastate (n={intra_vs_inter.loc['Intrastate','order_count']:,.0f}) vs "
             f"Interstate (n={intra_vs_inter.loc['Interstate','order_count']:,.0f}) orders",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3. Destination-state performance (all 27 states)
States with fewer than 20 eligible orders are flagged as low-sample; none are removed silently.

In [ ]:
state_perf = valid.groupby("customer_state").agg(
    orders=("order_id", "count"),
    delayed_orders=("delivery_outcome", lambda x: (x == "delayed").sum()),
    median_days=("total_delivery_duration_days", "median"),
    delay_pct=("delivery_outcome", delay_rate),
    avg_freight=("total_freight_value", "mean"),
).sort_values("delay_pct", ascending=False)
state_perf["low_sample"] = state_perf["orders"] < MIN_ORDERS

print(f"States analysed: {len(state_perf)} | states below {MIN_ORDERS}-order threshold: {int(state_perf['low_sample'].sum())}")
display(state_perf)

In [ ]:
plot_df = state_perf[~state_perf["low_sample"]].reset_index()
colors = [ACCENT if v > national_delay else NEUTRAL for v in plot_df["delay_pct"]]

fig, ax = plt.subplots(figsize=(15, 6))
bars = ax.bar(plot_df["customer_state"], plot_df["delay_pct"], color=colors)
ax.axhline(national_delay, color=PRIMARY, linestyle="--", linewidth=1.5, label=f"National average ({national_delay:.1f}%)")
for bar, n in zip(bars, plot_df["orders"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, f"n={n:,}", ha="center", fontsize=7.5, rotation=90)
ax.set_title("Delivery delay rate by destination state (red = above national average)", fontsize=13, fontweight="bold")
ax.set_xlabel("Customer destination state")
ax.set_ylabel("Delay rate (%)")
ax.set_ylim(0, plot_df["delay_pct"].max() * 1.2)
ax.legend()
plt.tight_layout()
plt.show()

## 4. The five most problematic shipping corridors
A corridor is an origin (seller) state → destination (customer) state pair, ranked by delay rate among corridors with **at least 20 orders**.

In [ ]:
corridor_all = valid.groupby(["seller_state", "customer_state"]).agg(
    order_count=("order_id", "count"),
    delayed_orders=("delivery_outcome", lambda x: (x == "delayed").sum()),
    median_days=("total_delivery_duration_days", "median"),
    delay_rate=("delivery_outcome", delay_rate),
    avg_freight=("total_freight_value", "mean"),
).reset_index()
corridor_all["corridor"] = corridor_all["seller_state"] + " → " + corridor_all["customer_state"]

corridor_perf = corridor_all[corridor_all["order_count"] >= MIN_ORDERS].copy()
print(f"Corridors observed: {len(corridor_all)} | with >= {MIN_ORDERS} orders: {len(corridor_perf)} "
      f"({corridor_perf['order_count'].sum() / len(valid) * 100:.1f}% of eligible orders)")

top_5_corridors = corridor_perf.sort_values("delay_rate", ascending=False).head(5)
display(top_5_corridors[["corridor", "order_count", "delayed_orders", "median_days", "delay_rate", "avg_freight"]]
        .reset_index(drop=True))

**Volume view.** A high delay *rate* on a small corridor matters less than a moderate rate on a large one, so the same corridors are also ranked by the *number* of delayed orders.

In [ ]:
top_by_volume = corridor_perf.sort_values("delayed_orders", ascending=False).head(5)
display(top_by_volume[["corridor", "order_count", "delayed_orders", "median_days", "delay_rate"]].reset_index(drop=True))

## 5. Regional delay matrix (origin × destination)

In [ ]:
top_dest = valid["customer_state"].value_counts().head(10).index
top_orig = valid["seller_state"].value_counts().head(10).index

mx = corridor_all[corridor_all["seller_state"].isin(top_orig) & corridor_all["customer_state"].isin(top_dest)].copy()
mx.loc[mx["order_count"] < MIN_ORDERS, "delay_rate"] = np.nan      # mask low-sample cells

rate_matrix = mx.pivot(index="seller_state", columns="customer_state", values="delay_rate").reindex(index=top_orig, columns=top_dest)
count_matrix = mx.pivot(index="seller_state", columns="customer_state", values="order_count").reindex(index=top_orig, columns=top_dest)

annot = rate_matrix.round(1).astype(str) + "\n(n=" + count_matrix.fillna(0).astype(int).astype(str) + ")"
annot = annot.where(rate_matrix.notna(), "n<20")

fig, ax = plt.subplots(figsize=(15, 9))
sns.heatmap(rate_matrix, annot=annot, fmt="", cmap="Reds", linewidths=0.5, linecolor="white",
            annot_kws={"fontsize": 8}, cbar_kws={"label": "Delay rate (%)"}, ax=ax)
ax.set_title("Regional shipping corridor delay matrix: top 10 origin × top 10 destination states\n"
             "Cells show delay rate % and order count; cells with n<20 are blank", fontsize=13, fontweight="bold")
ax.set_xlabel("Destination state (customer)")
ax.set_ylabel("Origin state (seller)")
plt.tight_layout()

out = FIG_DIR / "regional_delay_matrix.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out, "| exists:", out.exists())

## 6. Distance bands
The analytical dataset has no coordinates, so seller→customer distance is derived from the raw geolocation table (mean latitude/longitude per ZIP prefix) with the haversine formula. This is an approximation at ZIP-prefix level; orders whose ZIP prefix cannot be geocoded are excluded from this section only.

In [ ]:
valid_geo = add_distance_km(valid)
missing = valid_geo["distance_km"].isna().sum()
print(f"Orders without a computable distance: {missing:,} ({missing / len(valid_geo) * 100:.2f}%), excluded from this section")

valid_geo = valid_geo.dropna(subset=["distance_km"]).copy()
valid_geo["distance_band"] = pd.cut(valid_geo["distance_km"], bins=BAND_EDGES, labels=BAND_LABELS, right=False)
print(f"Median seller→customer distance: {valid_geo['distance_km'].median():,.0f} km")

In [ ]:
band_perf = valid_geo.groupby("distance_band", observed=True).agg(
    orders=("order_id", "count"),
    delayed_orders=("delivery_outcome", lambda x: (x == "delayed").sum()),
    median_days=("total_delivery_duration_days", "median"),
    delay_pct=("delivery_outcome", delay_rate),
    avg_freight=("total_freight_value", "mean"),
    median_weight_g=("total_weight_g", "median"),
)
display(band_perf)

corr = valid_geo[["distance_km", "total_delivery_duration_days", "total_freight_value"]].corr(method="spearman")
print("Spearman correlation with distance:")
print(f"  delivery duration : {corr.loc['distance_km', 'total_delivery_duration_days']:.2f}")
print(f"  freight value     : {corr.loc['distance_km', 'total_freight_value']:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
x = band_perf.index.astype(str)
for ax, col, title, color in zip(
        axes,
        ["median_days", "delay_pct", "avg_freight"],
        ["Median delivery duration (days)", "Delay rate (%)", "Average freight value (BRL)"],
        [PRIMARY, ACCENT, "#2E8B57"]):
    bars = ax.bar(x, band_perf[col], color=color)
    ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", rotation=40)
    ax.set_ylim(0, band_perf[col].max() * 1.15)
fig.suptitle("Delivery performance and freight cost by seller→customer distance band", fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
fig.savefig(FIG_DIR / "distance_band_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Key findings & caveats

In [ ]:
worst_state = state_perf[~state_perf["low_sample"]].iloc[0]
worst_state_name = state_perf[~state_perf["low_sample"]].index[0]
c1 = top_5_corridors.iloc[0]
rate_gap = intra_vs_inter.loc["Interstate", "delay_rate_pct"] - intra_vs_inter.loc["Intrastate", "delay_rate_pct"]
dur_ratio = intra_vs_inter.loc["Interstate", "median_duration_days"] / intra_vs_inter.loc["Intrastate", "median_duration_days"]

print(f"1. Interstate orders are {rate_gap:.1f} pp more likely to be late and take {dur_ratio:.1f}x longer (median) than intrastate orders.")
print(f"2. {worst_state_name} has the highest destination-state delay rate: {worst_state['delay_pct']:.1f}% "
      f"({int(worst_state['delayed_orders'])} of {int(worst_state['orders']):,} orders) vs {national_delay:.1f}% nationally.")
print(f"3. Highest-rate corridor (n>={MIN_ORDERS}): {c1['corridor']} at {c1['delay_rate']:.1f}% "
      f"({int(c1['delayed_orders'])} of {int(c1['order_count'])} orders).")
print(f"4. Delay rate rises from {band_perf['delay_pct'].iloc[0]:.1f}% (<100 km) to {band_perf['delay_pct'].iloc[-1]:.1f}% (>2,000 km); "
      f"median duration rises from {band_perf['median_days'].iloc[0]:.1f} to {band_perf['median_days'].iloc[-1]:.1f} days.")

**Caveats**
- Results are **associations**. Distance, carrier network quality, seller behaviour and product mix are intertwined, so no single factor is isolated here (this is addressed in the Week 3 root-cause notebook).
- Small corridors can show extreme rates from a handful of orders; corridors with fewer than 20 orders are excluded from rankings and matrix cells.
- Distance is approximate (ZIP-prefix centroids, great-circle rather than road distance).
- Delay compares actual delivery with the *promised* date, so regions with generous promise windows can look better than their true speed.